# Đồ án KTDL - Notebook xây mô hình nhanh từ domain features có sẵn

Notebook này phục vụ **Vai trò 3 - AI & Machine Learning (Hải Đăng)** theo flow trong file hướng dẫn đồ án.

Mục tiêu:
- nhận bộ dữ liệu đặc trưng từ Vai trò 2
- kiểm tra `labels` và `subject IDs`
- chia `train/test` theo **subject-level**
- xây baseline `FgMDM`
- xây stacked model từ nhiều domain features
- đánh giá bằng `Accuracy`, `F1`, `Sensitivity`, `Specificity`, `ROC-AUC`
- vẽ `Confusion Matrix` và `ROC Curve`


## 1. Mô tả flow của notebook

Notebook này dùng nhanh dữ liệu đã có sẵn trong folder `Super_MultiDomain_Features_Role3`.

Flow:
1. Nạp feature đã tính sẵn từ đầu ra Vai trò 2.
2. Kiểm tra phân bố nhãn và số epoch.
3. Chọn bài toán nhị phân.
4. Chia train/test theo subject để tránh data leakage.
5. Train baseline `FgMDM`.
6. Train stacked model.
7. Vẽ biểu đồ và lưu kết quả.

Notebook này phù hợp khi bạn muốn:
- train nhanh
- debug model
- viết phần trình bày cho Vai trò 3


In [ ]:
from pathlib import Path
import math
import os
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyriemann.classification import FgMDM, class_distinctiveness
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore', category=FutureWarning)


## 2. Cấu hình đường dẫn và tham số mô hình

Cell này gom các biến quan trọng để dễ đổi khi cần:
- thư mục project
- thư mục feature đã tính sẵn
- bài toán phân loại muốn chạy
- các band và metric muốn sử dụng


In [ ]:
ROOT = Path('/home/dohaidang/DataMining_Project')
PRECOMPUTED_DIR = ROOT / 'Super_MultiDomain_Features_Role3'
OUTPUT_DIR = ROOT / 'notebook_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
INNER_FOLDS = 5
FGMDM_METRIC = 'euclid'
FGMDM_FALLBACK_METRICS = ('riemann',)
FILTER_RATIO = 0.5
ELASTICNET_ALPHA = 0.01
ELASTICNET_L1_RATIO = 0.15

PROBLEMS = {
    'ad_hc': ('A', 'C', 'AD', 'HC'),
    'ftd_hc': ('F', 'C', 'FTD', 'HC'),
    'ftd_ad': ('F', 'A', 'FTD', 'AD'),
}

SELECTED_PROBLEM = 'ad_hc'
SELECTED_BANDS = {'delta', 'theta', 'alpha', 'beta', 'gamma'}
SELECTED_METRICS = {'cov', 'corr', 'plv', 'coh', 'csd', 'mi'}


## 3. Hàm nạp bộ domain features đã tính sẵn

Dữ liệu đầu vào của notebook này là output từ Vai trò 2:
- `feature `.npy` files`
- `labels.npy`
- `subject_ids.npy`
- cac file feature nhu `alpha_cov.npy`, `beta_plv.npy`, ...


In [ ]:
def normalize_subject_ids(subject_ids: np.ndarray) -> np.ndarray:
    normalized = []
    for subject_id in subject_ids.astype(str):
        if subject_id.startswith('sub-'):
            normalized.append(subject_id)
        else:
            normalized.append(f'sub-{subject_id.zfill(3)}')
    return np.asarray(normalized)


def nearest_spd_matrix(matrix: np.ndarray, jitter: float = 1e-6) -> np.ndarray:
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
    symmetric = 0.5 * (matrix + matrix.T)
    eigvals, eigvecs = np.linalg.eigh(symmetric)
    eigvals = np.maximum(eigvals, jitter)
    stabilized = (eigvecs * eigvals) @ eigvecs.T
    stabilized = 0.5 * (stabilized + stabilized.T)
    stabilized += np.eye(stabilized.shape[0]) * jitter
    return stabilized


def stabilize_spd_stack(matrices: np.ndarray, jitter: float = 1e-6) -> np.ndarray:
    return np.stack([nearest_spd_matrix(matrix, jitter=jitter) for matrix in matrices]).astype(np.float64)


def discover_feature_names(precomputed_dir: Path, selected_bands: set[str], selected_metrics: set[str]) -> list[str]:
    feature_names = []
    for path in sorted(precomputed_dir.glob('*.npy')):
        if path.stem in {'labels', 'subject_ids', 'top_features_name'}:
            continue
        if '_' not in path.stem:
            continue
        band, metric = path.stem.split('_', maxsplit=1)
        if band in selected_bands and metric in selected_metrics:
            feature_names.append(path.stem)
    if not feature_names:
        raise ValueError('No matching feature files found. Check selected bands/metrics and PRECOMPUTED_DIR.')
    return feature_names


def load_precomputed_feature_catalog(precomputed_dir: Path, selected_bands: set[str], selected_metrics: set[str]):
    feature_names = discover_feature_names(precomputed_dir, selected_bands, selected_metrics)
    labels = np.load(precomputed_dir / 'labels.npy', allow_pickle=True).astype(str)
    subject_ids = normalize_subject_ids(np.load(precomputed_dir / 'subject_ids.npy', allow_pickle=True))

    catalog = {}
    for feature_name in feature_names:
        band, metric = feature_name.split('_', maxsplit=1)
        matrices = np.load(precomputed_dir / f'{feature_name}.npy', allow_pickle=True).astype(np.float64)
        matrices = stabilize_spd_stack(matrices)
        catalog[f'{band}__{metric}'] = {
            'band': band,
            'metric': metric,
            'matrices': matrices,
            'labels': labels,
            'subject_ids': subject_ids,
            'spd_stabilized': True,
        }
    return catalog


## 4. Nạp feature catalog

Cell này nạp các feature dùng cho notebook hiện tại.


In [ ]:
feature_catalog = load_precomputed_feature_catalog(
    PRECOMPUTED_DIR,
    selected_bands=SELECTED_BANDS,
    selected_metrics=SELECTED_METRICS,
)

print('So feature da nap:', len(feature_catalog))
for key in sorted(feature_catalog):
    print(' -', key, feature_catalog[key]['matrices'].shape)


## 5. Khám phá dữ liệu nhanh (EDA nhỏ)

Bước này giúp kiểm tra:
- phân bố nhãn ở mức epoch
- số subject mỗi nhóm
- số epoch mỗi subject


In [ ]:
sample_feature = next(iter(feature_catalog.values()))
labels = sample_feature['labels']
subject_ids = sample_feature['subject_ids']

epoch_df = pd.DataFrame({'subject_id': subject_ids, 'label': labels})
subject_df = epoch_df.drop_duplicates('subject_id').reset_index(drop=True)
subject_df['epoch_count'] = subject_df['subject_id'].map(epoch_df['subject_id'].value_counts())

display(epoch_df['label'].value_counts().rename('epoch_count').to_frame())
display(subject_df['label'].value_counts().rename('subject_count').to_frame())
display(subject_df.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epoch_df['label'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#4C78A8', '#F58518', '#54A24B'])
axes[0].set_title('So epoch theo nhan')
axes[0].set_xlabel('Nhan')
axes[0].set_ylabel('So epoch')

subject_df['epoch_count'].plot(kind='hist', bins=20, ax=axes[1], color='#E45756')
axes[1].set_title('So epoch tren moi subject')
axes[1].set_xlabel('Epoch count')

plt.tight_layout()
plt.show()


## 6. Hàm hỗ trợ chia train/test và tính metric

Nhóm hàm này là xương sống của notebook:
- lọc bài toán nhị phân
- tạo bảng ở mức subject
- tính sample weight
- gộp xác suất từ epoch lên subject
- tính metric cuối cùng


In [ ]:
def binary_subset(feature_item, positive_code: str, negative_code: str):
    mask = np.isin(feature_item['labels'], [positive_code, negative_code])
    matrices = feature_item['matrices'][mask]
    if not feature_item.get('spd_stabilized', False):
        matrices = stabilize_spd_stack(matrices)
    labels = (feature_item['labels'][mask] == positive_code).astype(int)
    subjects = feature_item['subject_ids'][mask]
    return matrices, labels, subjects


def build_subject_frame(labels: np.ndarray, subject_ids: np.ndarray):
    frame = pd.DataFrame({'subject_id': subject_ids, 'label': labels})
    subject_frame = frame.drop_duplicates('subject_id').sort_values('subject_id').reset_index(drop=True)
    counts = frame.groupby('subject_id').size().rename('epoch_count')
    subject_frame = subject_frame.merge(counts, on='subject_id', how='left')
    subject_frame['sample_weight'] = 1.0 / subject_frame['epoch_count']
    return subject_frame


def epoch_weights(subject_ids: np.ndarray) -> np.ndarray:
    counts = pd.Series(subject_ids).value_counts()
    return np.asarray([1.0 / counts[sid] for sid in subject_ids], dtype=np.float64)


def aggregate_probs_by_subject(probabilities: np.ndarray, subject_ids: np.ndarray) -> pd.Series:
    frame = pd.DataFrame({'subject_id': subject_ids, 'probability': probabilities})
    return frame.groupby('subject_id')['probability'].mean()


def aggregate_labels_by_subject(labels: np.ndarray, subject_ids: np.ndarray) -> pd.Series:
    frame = pd.DataFrame({'subject_id': subject_ids, 'label': labels})
    return frame.groupby('subject_id')['label'].first()


def evaluate_subject_predictions(y_true: np.ndarray, y_prob: np.ndarray):
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics = {
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'sensitivity': float(tp / (tp + fn) if (tp + fn) else 0.0),
        'specificity': float(tn / (tn + fp) if (tn + fp) else 0.0),
    }
    return metrics, y_pred


def plot_confusion_and_roc(y_true: np.ndarray, y_prob: np.ndarray, title: str):
    y_pred = (y_prob >= 0.5).astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=axes[0], colorbar=False)
    axes[0].set_title(f'{title} - Confusion Matrix')

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)
    axes[1].plot(fpr, tpr, label=f'ROC-AUC = {auc_value:.4f}')
    axes[1].plot([0, 1], [0, 1], '--', color='gray')
    axes[1].set_title(f'{title} - ROC Curve')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def fit_fgmdm_with_fallback(X_train: np.ndarray, y_train: np.ndarray, sample_weight: np.ndarray):
    last_error = None
    for metric in (FGMDM_METRIC, *FGMDM_FALLBACK_METRICS):
        try:
            model = FgMDM(metric=metric)
            model.fit(X_train, y_train, sample_weight=sample_weight)
            return model, metric
        except ValueError as exc:
            last_error = exc
    raise last_error


def safe_class_distinctiveness(X: np.ndarray, y: np.ndarray):
    last_error = None
    for metric in (FGMDM_METRIC, *FGMDM_FALLBACK_METRICS):
        try:
            return float(class_distinctiveness(X, y, metric=metric)), metric
        except ValueError as exc:
            last_error = exc
    raise last_error


def predict_positive_proba(model, X: np.ndarray, positive_label: int = 1) -> np.ndarray:
    class_index = int(np.where(model.classes_ == positive_label)[0][0])
    return model.predict_proba(X)[:, class_index]


## 7. Chọn bài toán và chia train/test theo subject

Đây là phần quan trọng nhất để tránh leakage: **tất cả epoch của một subject chỉ nằm ở train hoặc test**, không được xuất hiện ở cả hai tập.


In [ ]:
positive_code, negative_code, positive_name, negative_name = PROBLEMS[SELECTED_PROBLEM]
reference_feature = next(iter(feature_catalog.values()))
_, y_binary, subject_ids_binary = binary_subset(reference_feature, positive_code, negative_code)
subject_table = build_subject_frame(y_binary, subject_ids_binary)

train_subject_ids, test_subject_ids = train_test_split(
    subject_table['subject_id'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=subject_table['label'],
)

train_subject_ids = np.asarray(sorted(train_subject_ids))
test_subject_ids = np.asarray(sorted(test_subject_ids))

train_subject_table = subject_table[subject_table['subject_id'].isin(train_subject_ids)].copy()
test_subject_table = subject_table[subject_table['subject_id'].isin(test_subject_ids)].copy()

print('Bai toan:', SELECTED_PROBLEM, f'({positive_name} vs {negative_name})')
print('So subject train:', len(train_subject_table))
print('So subject test:', len(test_subject_table))
display(train_subject_table['label'].value_counts().rename('train_subject_count').to_frame())
display(test_subject_table['label'].value_counts().rename('test_subject_count').to_frame())


## 8. Baseline - FgMDM trên một feature

Bước này xây mô hình cơ bản để làm mốc so sánh với stacked model.


In [ ]:
def run_fgmdm_single(feature_name: str, problem_name: str, train_subject_ids: np.ndarray, test_subject_ids: np.ndarray):
    positive_code, negative_code, positive_name, negative_name = PROBLEMS[problem_name]
    feature_item = feature_catalog[feature_name]
    X, y, subjects = binary_subset(feature_item, positive_code, negative_code)

    train_mask = np.isin(subjects, train_subject_ids)
    test_mask = np.isin(subjects, test_subject_ids)

    X_train, y_train, s_train = X[train_mask], y[train_mask], subjects[train_mask]
    X_test, y_test, s_test = X[test_mask], y[test_mask], subjects[test_mask]

    model, used_metric = fit_fgmdm_with_fallback(X_train, y_train, epoch_weights(s_train))

    test_epoch_prob = predict_positive_proba(model, X_test)
    subject_prob = aggregate_probs_by_subject(test_epoch_prob, s_test)
    subject_true = aggregate_labels_by_subject(y_test, s_test)

    subject_prob = subject_prob.reindex(sorted(subject_prob.index))
    subject_true = subject_true.reindex(subject_prob.index)
    metrics, subject_pred = evaluate_subject_predictions(subject_true.to_numpy(), subject_prob.to_numpy())

    return {
        'feature_name': feature_name,
        'fgmdm_metric_used': used_metric,
        'metrics': metrics,
        'subject_true': subject_true.to_numpy(),
        'subject_prob': subject_prob.to_numpy(),
        'subject_ids': subject_prob.index.to_numpy(),
    }


In [ ]:
BASELINE_FEATURE = sorted(feature_catalog.keys())[0]
baseline_result = run_fgmdm_single(
    feature_name=BASELINE_FEATURE,
    problem_name=SELECTED_PROBLEM,
    train_subject_ids=train_subject_ids,
    test_subject_ids=test_subject_ids,
)

pd.DataFrame([baseline_result['metrics']], index=[BASELINE_FEATURE])


In [ ]:
plot_confusion_and_roc(
    baseline_result['subject_true'],
    baseline_result['subject_prob'],
    title=f'Baseline FgMDM - {BASELINE_FEATURE}',
)


## 9. Stacked model từ nhiều domain features

Bước này bám sát Vai trò 3 hơn:
- xếp hạng feature bằng `class_distinctiveness`
- train base model `FgMDM` trên từng feature
- lấy xác suất ở mức epoch rồi average lên subject-level
- dùng logistic regression Elastic Net làm meta-model


In [ ]:
def safe_inner_folds(labels: np.ndarray, requested_folds: int) -> int:
    class_counts = np.bincount(labels.astype(int))
    min_count = int(class_counts.min())
    if min_count < 2:
        raise ValueError('At least two subjects per class are required for inner cross-validation.')
    return int(min(requested_folds, min_count))


def build_meta_model(alpha: float, l1_ratio: float, random_state: int) -> LogisticRegression:
    return LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        l1_ratio=l1_ratio,
        C=1.0 / alpha,
        max_iter=3000,
        random_state=random_state,
    )


def fit_base_subject_probabilities(feature_name: str, problem_name: str, train_subject_table: pd.DataFrame, test_subject_table: pd.DataFrame):
    positive_code, negative_code, _, _ = PROBLEMS[problem_name]
    feature_item = feature_catalog[feature_name]
    X, y, subjects = binary_subset(feature_item, positive_code, negative_code)

    train_subject_ids = set(train_subject_table['subject_id'])
    test_subject_ids = set(test_subject_table['subject_id'])
    train_mask = np.isin(subjects, list(train_subject_ids))
    test_mask = np.isin(subjects, list(test_subject_ids))

    X_train, y_train, s_train = X[train_mask], y[train_mask], subjects[train_mask]
    X_test, y_test, s_test = X[test_mask], y[test_mask], subjects[test_mask]

    train_subject_frame = build_subject_frame(y_train, s_train)
    inner_splits = safe_inner_folds(train_subject_frame['label'].to_numpy(), INNER_FOLDS)
    inner_cv = StratifiedGroupKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE)
    sample_weight = epoch_weights(s_train)

    oof_epoch_prob = np.empty(y_train.shape[0], dtype=np.float64)
    for inner_train_idx, inner_valid_idx in inner_cv.split(X_train, y_train, s_train):
        model, _ = fit_fgmdm_with_fallback(
            X_train[inner_train_idx],
            y_train[inner_train_idx],
            sample_weight[inner_train_idx],
        )
        oof_epoch_prob[inner_valid_idx] = predict_positive_proba(model, X_train[inner_valid_idx])

    oof_subject_prob = aggregate_probs_by_subject(oof_epoch_prob, s_train)
    oof_subject_prob = oof_subject_prob.reindex(train_subject_frame['subject_id']).to_numpy()

    final_model, _ = fit_fgmdm_with_fallback(X_train, y_train, sample_weight)
    test_epoch_prob = predict_positive_proba(final_model, X_test)
    test_subject_prob = aggregate_probs_by_subject(test_epoch_prob, s_test)
    test_subject_prob = test_subject_prob.reindex(test_subject_table['subject_id']).to_numpy()

    return oof_subject_prob, test_subject_prob


def meta_score(X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray, selected_columns: list[int]) -> float:
    n_splits = safe_inner_folds(y, INNER_FOLDS)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for train_idx, valid_idx in splitter.split(X[:, selected_columns], y):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X[train_idx][:, selected_columns])
        X_valid_scaled = scaler.transform(X[valid_idx][:, selected_columns])
        model = build_meta_model(ELASTICNET_ALPHA, ELASTICNET_L1_RATIO, RANDOM_STATE)
        model.fit(X_train_scaled, y[train_idx], sample_weight=sample_weight[train_idx])
        valid_prob = predict_positive_proba(model, X_valid_scaled)
        scores.append(roc_auc_score(y[valid_idx], valid_prob))
    return float(np.mean(scores))


def greedy_wrapper_selection(X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray) -> list[int]:
    selected = []
    remaining = list(range(X.shape[1]))
    best_score = -math.inf
    while remaining:
        trial_scores = []
        for candidate in remaining:
            score = meta_score(X, y, sample_weight, selected + [candidate])
            trial_scores.append((score, candidate))
        trial_scores.sort(reverse=True)
        candidate_score, candidate_idx = trial_scores[0]
        if not selected or candidate_score > best_score:
            selected.append(candidate_idx)
            remaining.remove(candidate_idx)
            best_score = candidate_score
        else:
            break
    return selected


In [ ]:
def run_stacked_model(problem_name: str, train_subject_table: pd.DataFrame, test_subject_table: pd.DataFrame):
    positive_code, negative_code, positive_name, negative_name = PROBLEMS[problem_name]

    distinctiveness_scores = {}
    for name, feature_item in feature_catalog.items():
        X, y, subjects = binary_subset(feature_item, positive_code, negative_code)
        train_mask = np.isin(subjects, train_subject_table['subject_id'])
        score, _ = safe_class_distinctiveness(X[train_mask], y[train_mask])
        distinctiveness_scores[name] = score

    ranked = sorted(distinctiveness_scores.items(), key=lambda item: item[1], reverse=True)
    retain_count = max(1, int(math.ceil(len(ranked) * FILTER_RATIO)))
    retained_features = [name for name, _ in ranked[:retain_count]]

    meta_columns_train = []
    meta_columns_test = []
    for feature_name in retained_features:
        train_prob, test_prob = fit_base_subject_probabilities(feature_name, problem_name, train_subject_table, test_subject_table)
        meta_columns_train.append(train_prob)
        meta_columns_test.append(test_prob)

    X_meta_train = np.column_stack(meta_columns_train)
    X_meta_test = np.column_stack(meta_columns_test)
    y_meta_train = train_subject_table['label'].to_numpy()
    y_meta_test = test_subject_table['label'].to_numpy()
    sample_weight = train_subject_table['sample_weight'].to_numpy(dtype=np.float64)

    selected_indices = greedy_wrapper_selection(X_meta_train, y_meta_train, sample_weight)
    if not selected_indices:
        selected_indices = [0]

    scaler = StandardScaler()
    X_meta_train_selected = scaler.fit_transform(X_meta_train[:, selected_indices])
    X_meta_test_selected = scaler.transform(X_meta_test[:, selected_indices])
    meta_model = build_meta_model(ELASTICNET_ALPHA, ELASTICNET_L1_RATIO, RANDOM_STATE)
    meta_model.fit(X_meta_train_selected, y_meta_train, sample_weight=sample_weight)
    y_prob = predict_positive_proba(meta_model, X_meta_test_selected)
    metrics, y_pred = evaluate_subject_predictions(y_meta_test, y_prob)

    return {
        'metrics': metrics,
        'retained_features': retained_features,
        'selected_features': [retained_features[idx] for idx in selected_indices],
        'subject_true': y_meta_test,
        'subject_prob': y_prob,
        'subject_ids': test_subject_table['subject_id'].to_numpy(),
    }


In [ ]:
stacked_result = run_stacked_model(
    problem_name=SELECTED_PROBLEM,
    train_subject_table=train_subject_table,
    test_subject_table=test_subject_table,
)

print('Feature giu lai sau filter step:')
print(stacked_result['retained_features'])
print('\nFeature cuoi cung sau wrapper step:')
print(stacked_result['selected_features'])

pd.DataFrame([stacked_result['metrics']], index=['stacked_model'])


In [ ]:
plot_confusion_and_roc(
    stacked_result['subject_true'],
    stacked_result['subject_prob'],
    title='Stacked Model',
)


## 10. So sánh baseline và stacked model

Bảng tổng hợp này rất hữu ích cho phần Results và Slide.


In [ ]:
comparison_df = pd.DataFrame(
    [baseline_result['metrics'], stacked_result['metrics']],
    index=[f'baseline::{BASELINE_FEATURE}', 'stacked_model'],
)
comparison_df


## 11. Lưu kết quả

Cell cuối cùng lưu metric và dự đoán ở mức subject để phục vụ viết báo cáo hoặc làm slide.


In [ ]:
comparison_path = OUTPUT_DIR / f'{SELECTED_PROBLEM}_comparison_metrics.csv'
baseline_pred_path = OUTPUT_DIR / f'{SELECTED_PROBLEM}_baseline_subject_predictions.csv'
stacked_pred_path = OUTPUT_DIR / f'{SELECTED_PROBLEM}_stacked_subject_predictions.csv'

comparison_df.to_csv(comparison_path)

pd.DataFrame({
    'subject_id': baseline_result['subject_ids'],
    'y_true': baseline_result['subject_true'],
    'y_prob': baseline_result['subject_prob'],
}).to_csv(baseline_pred_path, index=False)

pd.DataFrame({
    'subject_id': stacked_result['subject_ids'],
    'y_true': stacked_result['subject_true'],
    'y_prob': stacked_result['subject_prob'],
}).to_csv(stacked_pred_path, index=False)

print('Da luu:')
print(' -', comparison_path)
print(' -', baseline_pred_path)
print(' -', stacked_pred_path)
